<a href="https://colab.research.google.com/github/SpringBoard795/PicasoPhrase_Infosys_Internship_Nov2024/blob/MerajBegum/LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#LSTM

In [2]:
!pip install faker


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 19.1 MB/s eta 0:00:00


In [3]:
from faker import Faker
import random

In [4]:
fake=Faker()

In [5]:
texts=[]
labels=[]
num_samples=2000
for i in range(num_samples):
  if random.random()>0.5:
    texts.append(fake.text(max_nb_chars=50))
    labels.append(1)
  else:
    texts.append(fake.text(max_nb_chars=50))
    labels.append(0)

In [6]:
train_texts=texts[:1600]
test_texts=texts[1600:]
train_labels=labels[:1600]
test_labels=labels[1600:]

In [7]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [8]:
tokenizer=Tokenizer()
tokenizer.fit_on_texts(train_texts)

In [9]:
train_sequences=tokenizer.texts_to_sequences(train_texts)
test_sequences=tokenizer.texts_to_sequences(test_texts)

In [10]:
train_sequences

[[670, 127, 777, 304, 305, 540],
 [541, 306, 128, 75, 4],
 [862, 47, 76, 210, 77],
 [863, 864, 10, 129, 542, 24, 416, 778],
 [965, 211, 212, 671],
 [865, 307, 308],
 [130, 672, 213, 417, 25, 5],
 [418, 543, 76, 48, 214, 26, 673],
 [419, 309, 544, 131, 420, 27, 78],
 [421, 422, 215, 779, 310],
 [79, 80, 28, 311],
 [545, 312, 546, 313],
 [547, 313, 216, 132, 81],
 [82, 133, 423, 314, 217, 548],
 [1, 780, 83, 549, 781, 550, 84],
 [424, 29, 315, 85, 551, 425, 552],
 [553, 782, 218, 49],
 [783, 554, 674, 426],
 [555, 134, 135, 50, 316, 136],
 [866, 427, 556, 137, 317, 219, 428],
 [138, 783, 139, 557],
 [429, 784, 675, 220],
 [30, 558, 785, 867, 221],
 [212, 318, 86, 140, 670, 430],
 [559, 431, 676, 141, 142, 677, 432],
 [139, 222, 670, 11, 223, 224],
 [678, 868, 225, 129, 51],
 [226, 433, 542, 679, 319, 227],
 [87, 786, 88, 680, 214],
 [434, 143, 560, 787, 31],
 [869, 435, 228, 680, 540],
 [29, 788, 673, 229, 14],
 [144, 681, 561, 562, 682],
 [230, 49, 561, 308],
 [15, 89, 320, 789, 563, 88

In [11]:
word_index=tokenizer.word_index

In [12]:
max_sequence_length=max(len(seq) for seq in train_sequences)

In [13]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [14]:
max_sequence_length

10

In [15]:
train_padded=pad_sequences(train_sequences,maxlen=max_sequence_length,padding="post")
test_padded=pad_sequences(test_sequences,maxlen=max_sequence_length,padding="post")

In [16]:
test_padded

array([[556, 282, 299, ...,   0,   0,   0],
       [ 46, 108, 469, ...,   0,   0,   0],
       [450, 866, 612, ...,   0,   0,   0],
       ...,
       [392, 464, 944, ...,   0,   0,   0],
       [740, 260,  35, ...,   0,   0,   0],
       [110, 911, 813, ...,   0,   0,   0]], dtype=int32)

In [17]:
from tensorflow.keras.utils import to_categorical

In [18]:
train_labels=to_categorical(train_labels)
test_labels=to_categorical(test_labels)

In [19]:
train_labels ##[cat, dog]

array([[1., 0.],
       [0., 1.],
       [0., 1.],
       ...,
       [0., 1.],
       [0., 1.],
       [0., 1.]])

In [20]:
from gensim.models import Word2Vec
w2v_model=Word2Vec(sentences=[text.split() for text in train_texts],vector_size=100)

In [21]:
import numpy as np

In [22]:
embedding_dim=100
embedding_matrix=np.zeros((len(word_index)+1,embedding_dim))

In [23]:
for word,i in word_index.items():
  if word in w2v_model.wv:
    embedding_matrix[i]=w2v_model.wv[word]

In [24]:
embedding_matrix

array([[ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [-0.00912997,  0.00384931,  0.00533762, ..., -0.00362984,
        -0.00893592,  0.00439809],
       [ 0.00591873, -0.00284028, -0.00949631, ...,  0.00048991,
         0.00410974,  0.00245353],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ]])

In [25]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense
model=Sequential([
        Embedding(input_dim=len(word_index)+1,
                  output_dim=embedding_dim,
                  weights=[embedding_matrix],
        input_length=max_sequence_length,
        trainable=False),
        LSTM(128),
        Dense(2,activation="softmax")
])

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [26]:
model.compile(optimizer="adam",loss="categorical_crossentropy",metrics=["accuracy"])

In [27]:
model.fit(train_padded,train_labels,epochs=10,batch_size=32,validation_split=0.2)

Epoch 1/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.4765 - loss: 0.6939 - val_accuracy: 0.5219 - val_loss: 0.6926
Epoch 2/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.5169 - loss: 0.6928 - val_accuracy: 0.5219 - val_loss: 0.6924
Epoch 3/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.5208 - loss: 0.6924 - val_accuracy: 0.5219 - val_loss: 0.6923
Epoch 4/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - accuracy: 0.5156 - loss: 0.6924 - val_accuracy: 0.4781 - val_loss: 0.6934
Epoch 5/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.5592 - loss: 0.6908 - val_accuracy: 0.5281 - val_loss: 0.6910
Epoch 6/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.5718 - loss: 0.6811 - val_accuracy: 0.5188 - val_loss: 0.6937
Epoch 7/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.6103 - loss: 0.6619 - val_accuracy: 0.5125 - val_loss: 0.7057
Epoch 8/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.5988 - loss: 0.6645 - val_accuracy: 0.5000 - v

In [28]:
test_loss,test_accuracy=model.evaluate(test_padded,test_labels)

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4853 - loss: 0.7152


In [29]:
test_accuracy

0.4749999940395355

#Bidirectional

In [30]:
from tensorflow.keras.layers import Bidirectional
model=Sequential([
        Embedding(input_dim=len(word_index)+1,
                  output_dim=embedding_dim,
                  weights=[embedding_matrix],
        input_length=max_sequence_length,
        trainable=False),
        Bidirectional(LSTM(128)),
        Dense(2,activation="softmax")
])

#Stacked LSTM

In [31]:
model=Sequential([
        Embedding(input_dim=len(word_index)+1,
                  output_dim=embedding_dim,
                  weights=[embedding_matrix],
        input_length=max_sequence_length,
        trainable=False),
        LSTM(128),
        LSTM(64),
        Dense(2,activation="softmax")
])

##Dropout LSTM

In [32]:
model=Sequential([
        Embedding(input_dim=len(word_index)+1,
                  output_dim=embedding_dim,
                  weights=[embedding_matrix],
        input_length=max_sequence_length,
        trainable=False),
        LSTM(128,dropout=0.2,),
        Dense(2,activation="softmax")
])

##GRU

In [33]:
texts = [
    "The quick brown fox jumps over the lazy dog",
    "GRU models are great for sequential data",
    "Deep learning models can process large amounts of text data",
    "This is an example of large text data for text prediction tasks",
    "Machine learning and natural language processing are key areas of AI",
    "Artificial intelligence is transforming industries",
    "Natural language processing is a branch of AI",
    "Deep learning techniques enable computers to understand images and text",
    "Sequence models like RNN and GRU are effective for time series data",
    "The future of AI relies on the integration of machine learning and big data",
    "Big data and AI are driving innovation across various sectors",
    "Language models are used in translation and sentiment analysis tasks",
    "Understanding context is crucial in natural language understanding",
    "Neural networks are designed to recognize patterns in large datasets",
]
categories = [0, 1, 2, 1, 0, 1, 2, 0, 0, 1, 0, 1, 2, 0] # 0->postive 1->negative 2->neutral

In [34]:
tokenizer=Tokenizer(num_words=1000)
tokenizer.fit_on_texts(texts)
sequences=tokenizer.texts_to_sequences(texts)

In [35]:
max_length=max(len(seq) for seq in sequences)
max_length

14

In [36]:
padded_sequences=pad_sequences(sequences,maxlen=max_length,padding="post")

In [37]:
num_classes=3
categorical_labels=to_categorical(categories,num_classes)

In [38]:
categorical_labels

array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.],
       [0., 1., 0.],
       [1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.],
       [1., 0., 0.]])

In [39]:
embedding_dim=64


In [40]:
from tensorflow.keras.layers import GRU
model=Sequential([
    Embedding(input_dim=1000,output_dim=embedding_dim,input_length=max_length),
    GRU(128,dropout=0.2),
    Dense(num_classes,activation="softmax")
])

In [41]:
model.compile(optimizer="adam",loss="categorical_crossentropy",metrics=['accuracy'])

In [42]:
model.fit(padded_sequences,categorical_labels,epochs=14)

Epoch 1/14
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.0714 - loss: 1.1030
Epoch 2/14
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.3571 - loss: 1.0956
Epoch 3/14
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.4286 - loss: 1.0844
Epoch 4/14
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.5000 - loss: 1.0818
Epoch 5/14
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.5000 - loss: 1.0731
Epoch 6/14
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.5000 - loss: 1.0649
Epoch 7/14
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.5000 - loss: 1.0599
Epoch 8/14
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step - accuracy: 0.5000 - loss: 1.0655
Epoch 9/14
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step - accuracy: 0.5000 - loss: 1.0523
Epoch 10/14
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.5000 - loss: 1.0540
Epoch 11/14
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.5000 - loss: 1.0513
Epoch 12/14
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.5000 - loss: 1.0518
E

#LSTM Text Generation

In [43]:
corpus="""The quick brown fox jumps over the lazy dog",
    "GRU models are great for sequential data",
    "Deep learning models can process large amounts of text data",
    "This is an example of large text data for text prediction tasks",
    "Machine learning and natural language processing are key areas of AI",
    "Artificial intelligence is transforming industries",
    "Natural language processing is a branch of AI",
    "Deep learning techniques enable computers to understand images and text",
    "Sequence models like RNN and GRU are effective for time series data",
    "The future of AI relies on the integration of machine learning and big data",
    "Big data and AI are driving innovation across various sectors",
    "Language models are used in translation and sentiment analysis tasks",
    "Understanding context is crucial in natural language understanding",
    "Neural networks are designed to recognize patterns in large datasets"""

In [44]:
tokenizer.fit_on_texts([corpus])

In [45]:
vocab_size=len(tokenizer.word_index)+1
vocab_size

80

In [46]:
sequences=[]

In [47]:
for line in corpus.split("\n"):
  token_list=tokenizer.texts_to_sequences([line])[0]
  for i in range(1,len(token_list)):
    ngram_sequence=token_list[:i+1]
    sequences.append(ngram_sequence)

In [48]:
sequences

[[5, 24],
 [5, 24, 25],
 [5, 24, 25, 26],
 [5, 24, 25, 26, 27],
 [5, 24, 25, 26, 27, 28],
 [5, 24, 25, 26, 27, 28, 5],
 [5, 24, 25, 26, 27, 28, 5, 29],
 [5, 24, 25, 26, 27, 28, 5, 29, 30],
 [16, 6],
 [16, 6, 1],
 [16, 6, 1, 31],
 [16, 6, 1, 31, 12],
 [16, 6, 1, 31, 12, 32],
 [16, 6, 1, 31, 12, 32, 2],
 [17, 7],
 [17, 7, 6],
 [17, 7, 6, 33],
 [17, 7, 6, 33, 34],
 [17, 7, 6, 33, 34, 13],
 [17, 7, 6, 33, 34, 13, 35],
 [17, 7, 6, 33, 34, 13, 35, 3],
 [17, 7, 6, 33, 34, 13, 35, 3, 8],
 [17, 7, 6, 33, 34, 13, 35, 3, 8, 2],
 [36, 9],
 [36, 9, 37],
 [36, 9, 37, 38],
 [36, 9, 37, 38, 3],
 [36, 9, 37, 38, 3, 13],
 [36, 9, 37, 38, 3, 13, 8],
 [36, 9, 37, 38, 3, 13, 8, 2],
 [36, 9, 37, 38, 3, 13, 8, 2, 12],
 [36, 9, 37, 38, 3, 13, 8, 2, 12, 8],
 [36, 9, 37, 38, 3, 13, 8, 2, 12, 8, 39],
 [36, 9, 37, 38, 3, 13, 8, 2, 12, 8, 39, 18],
 [19, 7],
 [19, 7, 4],
 [19, 7, 4, 14],
 [19, 7, 4, 14, 10],
 [19, 7, 4, 14, 10, 20],
 [19, 7, 4, 14, 10, 20, 1],
 [19, 7, 4, 14, 10, 20, 1, 40],
 [19, 7, 4, 14, 10, 20,

In [49]:
max_sequence_len=max([len(seq) for seq in sequences])

In [50]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
sequences=pad_sequences(sequences,maxlen=max_sequence_len,padding="pre")

In [51]:
sequences

array([[ 0,  0,  0, ...,  0,  5, 24],
       [ 0,  0,  0, ...,  5, 24, 25],
       [ 0,  0,  0, ..., 24, 25, 26],
       ...,
       [ 0,  0,  0, ..., 77, 78, 15],
       [ 0,  0,  0, ..., 78, 15, 13],
       [ 0,  0,  0, ..., 15, 13, 79]], dtype=int32)

In [52]:
from tensorflow.keras.utils import to_categorical
X=sequences[:,:-1]
y=sequences[:,-1]
y=to_categorical(y,num_classes=vocab_size)

In [53]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense
model=Sequential([
    Embedding(vocab_size,50,input_length=max_sequence_len-1),
    LSTM(100,return_sequences=False),
    Dense(vocab_size,activation="softmax")

])

In [54]:
model.compile(loss="categorical_crossentropy",optimizer="adam",metrics=['accuracy'])

In [55]:
model.fit(X,y,epochs=100)

Epoch 1/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.0000e+00 - loss: 4.3813
Epoch 2/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.0448 - loss: 4.3701
Epoch 3/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.0837 - loss: 4.3546
Epoch 4/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.0593 - loss: 4.3303
Epoch 5/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.0509 - loss: 4.2796
Epoch 6/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.0529 - loss: 4.1583
Epoch 7/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.0518 - loss: 4.1376
Epoch 8/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0651 - loss: 4.0754
Epoch 9/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.0877 - loss: 4.0552
Epoch 10/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.0773 - loss: 4.0743
Epoch 11/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.0481 - loss: 3.9735
Epoch 12/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.0941 

In [56]:
import numpy as np
def generate_text(seed_text,next_words,max_sequence_len):
  for _ in range(next_words):
    token_list=tokenizer.texts_to_sequences([seed_text])[0]
    token_list=pad_sequences([token_list],maxlen=max_sequence_len-1,padding="pre")
    predicted=np.argmax(model.predict(token_list,verbose=0),axis=-1)
    output_word=""
    for word,index in tokenizer.word_index.items():
      if index==predicted:
        output_word=word
        break
    seed_text+=" "+output_word
  return seed_text

In [57]:
seed_text="AI is"
generate_text(seed_text,next_words=5,max_sequence_len=max_sequence_len)

'AI is transforming industries industries on the'

#Final Image Captioning

In [58]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Add
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
import pickle
from tensorflow.keras.layers import Input, Dropout, Add
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [59]:
def extract_features(image_path,model):
  img=tf.keras.preprocessing.image.load_img(image_path,target_size=(299,299))
  img=tf.keras.preprocessing.image.img_to_array(img)
  img=np.expand_dims(img,axis=0)
  img=preprocess_input(img)
  features=model.predict(img)
  return features


In [60]:
image_model=InceptionV3(weights="imagenet")
image_model=Model(inputs=image_model.input,outputs=image_model.layers[-2].output)

96112376/96112376 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [61]:
def preprocess_captions(captions):
  tokenizer=Tokenizer()
  tokenizer.fit_on_texts(captions)
  sequences=tokenizer.texts_to_sequences(captions)
  max_length=max(len(seq) for seq in sequences)
  vocab_size=len(tokenizer.word_index)+1
  return tokenizer,sequences,max_length,vocab_size

In [70]:
image_captions = {
    "/content/image1.jpg": ["A cat on a bed"],
    "/content/image2.jpg": ["A dog running in a park"],
}

In [71]:
all_captions=[caption for captions in image_captions.values() for caption in captions]
tokenizer,sequences,max_length,vocab_size=preprocess_captions(all_captions)

In [72]:
sequences

[[1, 2, 3, 1, 4], [1, 5, 6, 7, 1, 8]]

In [73]:
image_features={}

In [74]:
for image_path in image_captions:
  image_features[image_path]=extract_features(image_path,image_model)

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 253ms/step


In [75]:
image_features

{'/content/image1.jpg': array([[0.0127638 , 0.7938619 , 0.24146496, ..., 0.29548547, 0.1911611 ,
         0.05291744]], dtype=float32),
 '/content/image2.jpg': array([[0.19318642, 0.05894864, 0.15388344, ..., 0.23879993, 0.6165901 ,
         0.13502048]], dtype=float32)}

In [76]:
def create_model(vocab_size,max_length):
  image_input=Input(shape=(2048,),name="image_input")
  image_dense=Dense(256,activation="relu",name="image_dense")(image_input)

  text_input=Input(shape=(max_length,))
  text_embedding=Embedding(vocab_size,256,mask_zero=True)(text_input)
  text_lstm=LSTM(256)(text_embedding)

  decoder=Add()([image_dense,text_lstm])
  decoder_dense=Dense(256,activation="relu")(decoder)
  output=Dense(vocab_size,activation="softmax")(decoder_dense)
  model=Model(inputs=[image_input,text_input],outputs=output)
  return model

In [77]:
model = create_model(vocab_size,max_length)
model.compile(loss="categorical_crossentropy",optimizer="adam")

In [78]:
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences

def data_generator(image_features, image_captions, tokenizer, max_length, vocab_size):
    while True:
        for image_id, captions in image_captions.items():
            image_feature = image_features[image_id][0]  # Extract feature vector
            for caption in captions:
                seq = tokenizer.texts_to_sequences([caption])[0]
                for i in range(1, len(seq)):
                    input_seq = seq[:i]
                    output_word = seq[i]

                    # Pad the input sequence
                    input_seq = pad_sequences([input_seq], maxlen=max_length, padding='post')[0]

                    # Convert to tensors
                    image_feature_tensor = tf.convert_to_tensor(image_feature, dtype=tf.float32)
                    input_seq_tensor = tf.convert_to_tensor(input_seq, dtype=tf.float32)
                    output_word_tensor = tf.keras.utils.to_categorical([output_word], num_classes=vocab_size)[0]
                    output_word_tensor = tf.convert_to_tensor(output_word_tensor, dtype=tf.float32)

                    yield (image_feature_tensor, input_seq_tensor), output_word_tensor


In [79]:
dataset = tf.data.Dataset.from_generator(
    lambda: data_generator(image_features, image_captions, tokenizer, max_length, vocab_size),
    output_signature=(
        (
            tf.TensorSpec(shape=(2048,), dtype=tf.float32),  # Image feature vector
            tf.TensorSpec(shape=(max_length,), dtype=tf.float32)  # Input sequence
        ),
        tf.TensorSpec(shape=(vocab_size,), dtype=tf.float32)  # Target one-hot vector
    )
)


In [82]:
batch_size = 64
train_dataset = dataset.batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)